# 02 - Backtest a trained run

Pick a strategy, edit its knobs and the costs, press **Run backtest**. Everything is replayed on the run's own
TEST block (out of sample: the purged split is rebuilt from the run's config) with next-open fills, stops on the
bar's high/low, fees + spread + slippage, and three baselines (buy-and-hold, always-flat, random entries at the
same frequency, holding time and size).

The dashboard stacks price and trades, the per-horizon P(up) against the strategy's entry lines, confidence and
signal strength, predicted sigma, equity and drawdown on one time axis; hover shows every panel at that bar. The
whole block shows the equity story; stops, holding periods and entry-to-exit lines are drawn on views of 800 bars
or fewer, so a detail window follows (its numbers are the window's own). Then the per-trade analytics.

In [1]:
# Parameters
RUN_DIR = None                  # a runs/<id> directory; None -> the newest run under RUNS_DIR
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
DETAIL_BARS = 600
DETAIL_AROUND = "steepest_fall" # "steepest_fall", "worst_trade" or "last"

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from neural_trade.notebook import BacktestExplorer, pick_run
from neural_trade.visualization.trading_dashboard import detail_window

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
explorer = BacktestExplorer.from_run(run_dir, csv_path=CSV_PATH)
display(explorer.widget())
explorer.click_run()   # render the default strategy once; then use the controls

run: ..\runs\20260924T165923Z-fa75177-dirty-af67ee43
CalibrationPipeline loaded from '..\runs\20260924T165923Z-fa75177-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


Static copy of that first result (the widget above holds the live one): the summary with the random null, the whole block, a detail window, and every trade.

In [3]:
display(explorer.summary_frame().round(4))
explorer.dashboard().show()
start, end = detail_window(explorer.last, DETAIL_BARS, around=DETAIL_AROUND)
explorer.dashboard(start=start, end=end).show()
explorer.trade_analytics().show()

,n_trades,total_return,sharpe_net,max_drawdown,hit_rate,hit_rate_gross,profit_factor,avg_win,avg_loss,expectancy,...,net_long,net_short,exposure,fees_paid,costs_paid,gross_return,random percentile (return),random percentile (gross),random p05 (return),random p95 (return)
calibrated_quantile,167.0,-0.3338,-91.3156,0.3338,0.1198,0.6048,0.0519,9.1369,-23.9531,-19.9903,...,-1528.4409,-1809.9323,0.1819,2733.3653,3553.3749,0.0215,73.0,84.0,NaN,NaN
buy_and_hold,1.0,0.0410,6.6470,0.0495,1.0000,1.0000,inf,NaN,NaN,NaN,...,NaN,NaN,0.9999,20.4362,26.5671,NaN,NaN,NaN,NaN,NaN
always_flat,0.0,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN
"random same freq (mean of 100 seeds, size 1.00)",NaN,-0.3496,-112.8638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0015,NaN,NaN,-0.386,-0.3079


## Every strategy with its default knobs

Each strategy is compared with random entries at its own trade rate, holding time and position size (the
diamonds; the same null as the summary table above). After costs the return is mostly cost x trade count, so only
the rank against that null tells skill from chance.

In [4]:
runs, comparison = explorer.compare_strategies()
comparison.show()
explorer.comparison_table(runs).round(4)

,n_trades,total_return,sharpe_net,max_drawdown,hit_rate,hit_rate_gross,profit_factor,exposure,gross_pnl,costs_paid,...,net_long,n_short,gross_short,net_short,random mean return,random p05 (return),random p95 (return),random percentile (return),random percentile (gross),exits
calibrated_quantile,167,-0.333837,-91.315608,0.333837,0.11976,0.60479,0.051898,0.181868,215.001722,3553.374948,...,-1528.440887,91,101.299422,-1809.932339,-0.349589,-0.38599,-0.307917,73.0,84.0,"REV 132, TIME 30, SL 5"
enhanced_multi_horizon,0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,...,0.0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,
liberal,102,-0.231512,-90.549658,0.231512,0.0,0.392157,0.0,0.030404,16.242048,2331.366397,...,0.0,102,16.242048,-2315.12435,-0.233553,-0.272987,-0.200781,48.0,59.0,"SL 62, TP 23, TP1 17"
threshold_spike,0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,...,0.0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,
buy_and_hold,1,0.040967,6.64696,0.049498,1.0,1.0,inf,0.999862,436.235136,26.567106,...,409.66803,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,EOW 1
always_flat,0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,...,0.0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,


A strategy with 0 trades is usually blocked by one of two things. Fixed probability lines
(`threshold_spike` enters above 0.65 / below 0.35) are rarely crossed by calibrated probabilities.
Strategies that need the predicted move to agree with the side (`enhanced_multi_horizon`) cannot trade
when delta shrinkage serves a zero delta. Delta shrinkage sets beta = 0 when the raw price head's
moves pointed the wrong way on the calibration block (negative correlation with the realised move).
This run's values:

In [5]:
p_up = explorer.signals.p
pd.DataFrame({"delta beta (served delta = beta x raw)": pd.Series(explorer.blocks["predictor"].bundle.calibration_pipeline.delta_scale),
              "P(up) 1st percentile": np.percentile(p_up, 1, axis=0),
              "P(up) 99th percentile": np.percentile(p_up, 99, axis=0)}, index=["h0", "h1", "h2"]).round(4)

,delta beta (served delta = beta x raw),P(up) 1st percentile,P(up) 99th percentile
h0,0.0,0.3876,0.5865
h1,0.0,0.3774,0.5951
h2,0.0,0.3923,0.5744
